# Recreate an image with a mosaic of images

## Notes:
- Here, this script will work better with images with same aspect ratio
- ToDo: make a script that can create a grid with images from different aspect ratio

## Import modules

In [10]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
# from pathlib import Path
from datetime import date
# from dataclasses import dataclass, field

# import 3rd-party modules
import cv2
import numpy as np
from numba import njit
# from pygifsicle import optimize


# import local modules
# from utils.renderer.giffer import create_gif
# from utils.renderer.videographer import create_video
from utils.project_manager import Project
# from utils.renderer.resizer import resize_with_pad, resize_with_crop

## Set up project

In [11]:
# name out img dirs
out_img_dir_list = ["mosaic"]

# create project
project = Project(project_dir="assets/images/mosaic/snapshots/spiderman", in_img_dir="db", out_img_dir_list=out_img_dir_list)

## Define functions and classes

In [16]:
# decorate function with numba fct to speed up execution
@njit()
def create_mosaic(src_imgs, dest_img, cell_shape, nb_cols, nb_rows):
    """
    Function to find best matching image for region of interest of another image
    Important: The images should be in LAB color space for better results.

    Arguments
    * src_img: source image
    * dest_img: destination image, i.e. image to recreate with images
    """
    # unpack grid cell image shape
    cell_height, cell_width = cell_shape[:2]
    
    # create empty lists (or arrays) to store coords mapping between source and dest images
    src_imgs_idxs_grid_yxs = []

    # get list of grid positions
    grid_positions = [(grid_y, grid_x) for grid_y in range(nb_cols) for grid_x in range(nb_rows)]

    # get list of grid positions
    grid_positions = [(grid_y, grid_x) for grid_y in range(nb_cols) for grid_x in range(nb_rows)]

    # get index range of grid positions
    grid_positions_idxs = np.arange(len(grid_positions))
    # choose random seed to recreate same shuffle or change it to see if you get better results
    np.random.seed(1111)
    # shuffle grid positions (otherwise the first grids from top will get the best matching images)
    np.random.shuffle(grid_positions_idxs)

    # iterate over each grid position
    for i in grid_positions_idxs:

        grid_y, grid_x = grid_positions[i]
        
        # get region of interest in img to recreate
        y = grid_y * cell_height
        x = grid_x * cell_width
        dest_roi = dest_img[y:y+cell_height, x:x+cell_width]

        # inititiate trackers for best match to this roi
        best_match_dist = np.inf
        best_match_index = 0

        # iterate over each image in list of source images
        for src_img_idx, src_img in enumerate(src_imgs):

            # compute distance between the pixels
            dist = np.sum(np.abs(dest_roi - src_img))
            
            # if distance is smaller than the current best dist
            if dist < best_match_dist:
                # update current best dist
                best_match_dist = dist
                # store best match index 
                best_match_index = src_img_idx

        # take out best match image from list of source images
        best_match_img = src_imgs.pop(best_match_index)

        # append best match index and its corresponding grid position to list
        src_imgs_idxs_grid_yxs.append((best_match_index, grid_y, grid_x))

        # update roi with best match image
        dest_img[y:y + ref_img_height, x:x + ref_img_width] = best_match_img

    return dest_img, src_imgs_idxs_grid_yxs

## Define variables & constants

In [17]:
# set grid caracteristics
NB_ROWS = 200
NB_COLS = 200

# set dest image path (i.e path of image to recreate)
dest_img_path = "assets/images/mosaic/snapshots/spiderman/to_recreate/spiderman_road.png"

## Read images

In [18]:
# read first image to get a reference shape
ref_img = cv2.imread(project.in_img_path_list[0])
ref_img_height, ref_img_width, ref_img_channel = ref_img.shape
# set new ref height & width to save memory if necessary
ref_img_height, ref_img_width = ref_img_height//1, ref_img_width//1

# set output grid image shape
out_img_height = ref_img_height * NB_ROWS
out_img_width = ref_img_width * NB_COLS
out_img_channel = ref_img_channel

# read images to use to recreate dest image
src_imgs = [cv2.resize(cv2.cvtColor(cv2.imread(src_img_path), cv2.COLOR_BGR2LAB), (ref_img_width, ref_img_height)) for src_img_path in project.in_img_path_list]

# get dest image and resize to output grid image shape
dest_img = cv2.resize(cv2.cvtColor(cv2.imread(dest_img_path), cv2.COLOR_BGR2LAB), (out_img_width, out_img_height))

## Recreate image

In [19]:
out_img, src_imgs_idxs_grid_yxs = create_mosaic(src_imgs, dest_img, cell_shape=(ref_img_height, ref_img_width), nb_cols=NB_COLS, nb_rows=NB_ROWS)

# convert image to bgr
out_img = cv2.cvtColor(out_img, cv2.COLOR_LAB2BGR)

# set output image directory
out_img_dir = "mosaic"

# get current date
today = date.today().strftime("%Y%m%d")

# set output image path
out_img_path = str(project.out_img_dir_dict[out_img_dir] / f"mosaic_{today}.jpg")

# save output image & metadata
cv2.imwrite(out_img_path, out_img)

/Users/derrickvanfrausum/anaconda3/envs/cv/lib/python3.9/site-packages/numba/core/ir_utils.py:2119: NumbaPendingDeprecationWarning: 
Encountered the use of a type that is scheduled for deprecation: type 'reflected list' found for argument 'src_imgs' of function 'create_mosaic'.

For more information visit https://numba.pydata.org/numba-doc/latest/reference/deprecation.html#deprecation-of-reflection-for-list-and-set-types

File "<ipython-input-16-272eb5ffc3db>", line 3:
@njit()
def create_mosaic(src_imgs, dest_img, cell_shape, nb_cols, nb_rows):
^

  warnings.warn(NumbaPendingDeprecationWarning(msg, loc=loc))


True

## 